## Reverb probing

The regression analysis of chapter 02 repeated under reverberation, sweeping C50 from −5 to 25 dB on a MUSE checkpoint fine-tuned for reverb (PESQ 2.17 to 3.02, STOI 0.834 to 0.944). The same depth-dependent progression of slopes and intercepts is recovered, with a compressed dynamic range relative to the noise setting.

The figure cells below run from the precomputed CKA parquet and are correct on any laptop. A bottom cell loads the model for live inference and is gated by `SE_PROBE_RUN_INFERENCE=1`. The fine-tuned checkpoint is fetched from HuggingFace by `scripts/setup.py`; the training pipeline that produced it lives at <https://github.com/YairAmar/muse-dereverb-ft> (also vendored under `training/`).

Runtime: figure cells under 30 seconds; live inference about 1 minute on CUDA, 5 minutes on MPS.


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO.name != "SE-Probe" and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from se_probe.device import device_info, get_device
from se_probe.plotting import MODEL_COLORS, MODEL_LABELS, apply_paper_rcparams

apply_paper_rcparams()

DEVICE = get_device()
print(device_info(DEVICE))

import pandas as pd

DATA_DIR = REPO / ("results_df" if (REPO / "results_df").exists() else "results_demo")
print(f"Using data from: {DATA_DIR}")

## Load reverb CKA table

C50 is the early-to-late energy ratio of the room impulse response (in dB); higher = drier. We restrict to the `[-5, 25]` C50 window where the linear fit makes sense and average across utterances.

In [ ]:
reverb_path = DATA_DIR / ("reverb/cka_reverb_muse_epoch_48.parquet" if (DATA_DIR / "reverb").exists() else "cka_reverb_muse_demo.parquet")
df = pd.read_parquet(reverb_path)
df = df[df["layer"].str.endswith(".norm1")].copy()
df = df[(df["c50"] >= -5) & (df["c50"] <= 25)]

block_order = ["encoder_level1", "encoder_level2", "latent", "decoder_level2", "decoder_level1", "mag_refinement"]
norm1_layers = sorted(df["layer"].unique(),
                       key=lambda L: (next((i for i, b in enumerate(block_order) if b in L), len(block_order)), L))
print(f"norm1 layers: {len(norm1_layers)} | rows after C50 filter: {len(df):,}")

## Per-layer slope of CKA vs C50

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

rows = []
for layer in norm1_layers:
    g = df[df["layer"] == layer]
    cell = g.groupby("c50", as_index=False)["CKA"].mean().sort_values("c50")
    x, y = cell["c50"].to_numpy(dtype=float), cell["CKA"].to_numpy(dtype=float)
    if len(np.unique(x)) < 2:
        continue
    reg = LinearRegression().fit(x.reshape(-1, 1), y)
    rows.append({"layer": layer, "slope": reg.coef_[0], "intercept": reg.intercept_,
                  "r2": reg.score(x.reshape(-1, 1), y)})
trend = pd.DataFrame(rows)
print(f"layers fit: {len(trend)} | mean R²: {trend['r2'].mean():.3f}")
trend.head()

In [ ]:
import matplotlib.pyplot as plt

block_size = max(1, len(trend) // 6)
tick_labels = ["Enc-L1", "Enc-L2", "Latent", "Dec-L2", "Dec-L1", "Refinement"]
tick_positions = [i * block_size + block_size / 2 - 0.5 for i in range(6)]
x = np.arange(len(trend))

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(x, trend["slope"].values, marker="o", color=MODEL_COLORS["muse"],
        label=f"{MODEL_LABELS['muse']} (reverb FT)")
ax.axhline(0, color="k", linestyle=":", linewidth=0.8)
ax.set_xlabel("Block")
ax.set_ylabel(r"d CKA / d C50 (dB$^{-1}$)")
ax.set_title("C50 sensitivity per norm1 layer")
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=30, ha="center")
for i in range(block_size, len(trend), block_size):
    ax.axvline(x=i - 0.5, color="k", linestyle="--", linewidth=0.6)
ax.legend(loc="best")
plt.tight_layout();

---

## Optional: live inference with the reverb fine-tuned MUSE

The cell below loads the epoch-48 fine-tuned checkpoint shipped via `scripts/setup.py` (HuggingFace `yairamr/SE-Probe-models`) and runs the probing pipeline on a single utterance. Training-from-scratch lives at <https://github.com/YairAmar/muse-dereverb-ft> (also vendored at `training/` if you cloned with `--recurse-submodules`). Set `SE_PROBE_RUN_INFERENCE=1` to enable it.


In [ ]:
import os

if os.environ.get("SE_PROBE_RUN_INFERENCE") == "1":
    import numpy as np

    from se_probe.activation_extraction import load_muse_activation_extractor_reverb
    from se_probe.cka import cka

    canonical = REPO / "pretrained_models/muse/muse_reverb_e48.pt"
    staging = REPO / "_workspace/release_artifacts/muse_reverb_e48.pt"
    ckpt = canonical if canonical.exists() else staging
    if not ckpt.exists():
        raise FileNotFoundError(
            "Reverb checkpoint not found. Run `python scripts/setup.py` to fetch it from HuggingFace."
        )
    print(f"Loading reverb FT MUSE from: {ckpt}")

    extractor = load_muse_activation_extractor_reverb(device=DEVICE, checkpoint_path=str(ckpt))

    rng = np.random.default_rng(0)
    sr = 16000
    clean = (rng.standard_normal(sr * 2) * 0.05).astype("float32")
    rir = (rng.standard_normal(int(sr * 0.5)) * 0.05).astype("float32")
    rir[0] = 1.0
    rev = np.convolve(clean, rir, mode="same").astype("float32")

    act_clean, _ = extractor(clean)
    act_rev, _ = extractor(rev)
    layer = next(iter(act_clean))
    val = cka(act_clean[layer], act_rev[layer])
    print(f"Live CKA on synthetic reverb clip ({layer}): {float(val):.4f}")
else:
    print("SE_PROBE_RUN_INFERENCE not set — skipping live inference. Set it to '1' to enable.")